# KERN-Lab-0.5-624M in Google Colab

Run the cells in order. A GPU runtime is optional (Runtime → Change runtime type → GPU). The model completes text prefixes. It is an experimental adaptation of Qwen3-0.6B-Base.

The notebook downloads approximately 1.3 GB. Review the custom model Python file in the downloaded snapshot before running the loading cell. It reconstructs the sparse layer through Transformers AutoClass.

In [ ]:
%pip -q install "torch>=2.7.0" "transformers==5.23.0" "huggingface_hub>=1.0" "safetensors>=0.4.0" "tokenizers>=0.20.0"

In [ ]:
from pathlib import Path
from hashlib import file_digest
from huggingface_hub import snapshot_download

MODEL_REPO = "VAL2409/KERN-Lab-0.5-624M"  # @param {type: "string"}
REVISION = ""  # @param {type: "string"}
LOCAL_MODEL_DIR = ""  # @param {type: "string"}

required = [
    "config.json", "generation_config.json", "model.safetensors",
    "modeling_kern_qwen06b_v1.py", "tokenizer.json", "tokenizer_config.json",
    "merges.txt", "vocab.json",
]
if LOCAL_MODEL_DIR.strip():
    model_dir = Path(LOCAL_MODEL_DIR).expanduser().resolve()
else:
    repo_id = MODEL_REPO.strip()
    if not repo_id or repo_id.startswith("__"):
        repo_id = input("Enter the official Hugging Face repository (owner/KERN-Lab-0.5-624M): ").strip()
    model_dir = Path(snapshot_download(
        repo_id=repo_id, revision=REVISION.strip() or None, allow_patterns=required,
    ))
with (model_dir / "model.safetensors").open("rb") as stream:
    actual_sha256 = file_digest(stream, "sha256").hexdigest()
expected_sha256 = "cc1ac73225ccc6013b1e73d91a6299078f7acd1b67940bf5277328dac43b0fd2"
if actual_sha256 != expected_sha256:
    raise ValueError(f"Unexpected SafeTensors SHA-256: {actual_sha256}")
print("Approved weights verified.")

In [ ]:
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
config = AutoConfig.from_pretrained(model_dir, trust_remote_code=True, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(model_dir, config=config, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    model_dir, config=config, trust_remote_code=True, local_files_only=True, dtype=dtype,
).to(device).eval()
print(f"Loaded {type(model).__name__} on {device} ({dtype}).")

In [ ]:
prompt = "The capital of France is"  # @param {type: "string"}
new_tokens = 16  # @param {type: "integer"}

inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
with torch.inference_mode():
    output = model.generate(
        **inputs, max_new_tokens=int(new_tokens), do_sample=False, use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
    )
print(prompt + tokenizer.decode(output[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True))